# Metamodelo de la funcion de gananciaSurrogate model entrenado sobre los resultados del Grid Search `HT29003` (rpart cp=-1).Objetivo: entender como se comporta la ganancia a traves del espacio de hiperparametros <maxdepth, minsplit, minbucket>.

### Seteo del ambiente local

In [ ]:
if (basename(getwd()) == "arboles") {  ROOT_DIR <- normalizePath(file.path(getwd(), "../.."), mustWork = TRUE)} else {  ROOT_DIR <- normalizePath(getwd(), mustWork = TRUE)}DATA_DIR <- file.path(ROOT_DIR, "datasets")EXP_DIR  <- file.path(ROOT_DIR, "exp")

Los resultados del Grid Search de la corrida anterior estan en`exp/HT29003/gridsearch_detalle.txt` (un registro por <semilla, parametros>).El metamodelo trabaja sobre la **ganancia media por combinacion** (promedio de las 5 semillas)para separar el ruido de semilla del comportamiento sistematico de la funcion objetivo.

In [ ]:
require("data.table")require("rpart")if (!require("rpart.plot")) install.packages("rpart.plot")require("rpart.plot")

In [ ]:
experimento <- "METAMODELO"dir.create(file.path(EXP_DIR, experimento), showWarnings = FALSE)setwd(file.path(EXP_DIR, experimento))

In [ ]:
# =============================================================================# 1. Cargar resultados del Grid anterior (HT29003)# =============================================================================archivo_prev <- file.path(EXP_DIR, "HT29003", "gridsearch_detalle.txt")if (!file.exists(archivo_prev)) stop("No encuentro los resultados previos en ", archivo_prev)tb_detalle <- fread(archivo_prev, sep = "\t")cat("Registros detalle :", nrow(tb_detalle), "\n")cat("Regimenes de cp   :\n"); print(table(tb_detalle$cp))

In [ ]:
# =============================================================================# 2. Agregar por combinacion de hiperparametros (media y desviacion de semillas)# =============================================================================tb_combo <- tb_detalle[  , .(ganancia_mean = mean(ganancia_test, na.rm = TRUE),      ganancia_sd   = sd(ganancia_test, na.rm = TRUE),      n_seeds       = .N),  by = .(cp, maxdepth, minsplit, minbucket)][order(-ganancia_mean)]cat("Combinaciones unicas:", nrow(tb_combo), "\n")tb_combo# El regimen cp ~ 0 produce arboles degenerados (ganancia 0) y domina la# importancia. Truco: separar la funcion en el regimen util cp = -1.tb_combo_neg <- tb_combo[cp == -1]cat("\nCombinaciones en el regimen cp=-1 (util):", nrow(tb_combo_neg), "\n")

El `minsplit` muestra una relacion bastante monotona con la ganancia media(cuanto mayor, mejor). El metamodelo intentara explicar la funcion gananciaa partir de los 4 hiperparametros.

In [ ]:
# =============================================================================# 3. Metamodelo: arbol ANOVA de ganancia_mean ~ hiperparametros# =============================================================================modelo_meta <- rpart(  ganancia_mean ~ cp + maxdepth + minsplit + minbucket,  data = tb_combo_neg,  method = "anova",  control = rpart.control(    cp = -1,                # permite explorar todo el espacio    minsplit  = 5L,         # pocas combinaciones: permitir splits finos    minbucket = 3L,    maxdepth  = 5  ))# Importancia relativa de cada hiperparametroimp <- modelo_meta$variable.importancecat("=== importancia de variables ===\n")print(round(100 * imp / sum(imp), 1))cat("\nR2 pred  :", 1 - modelo_meta$cptable[nrow(modelo_meta$cptable), "rel error"], "\n")print(modelo_meta$cptable)

In [ ]:
# Reglas del arbol metamodelocat("=== Reglas de division del metamodelo ===\n")print(as.data.frame(modelo_meta$splits))

In [ ]:
# Grafico del arbol metamodelopdf("arbol_metamodelo.pdf", width = 40, height = 30)prp(modelo_meta,    type = 4, extra = 101, branch = 1, digits = -5, varlen = 0, faclen = 0)dev.off()

In [ ]:
# =============================================================================# 4. Prediccion sobre una grilla densa (candidatos mas probables de ganar)# =============================================================================candidatos <- expand.grid(  maxdepth = seq(4, 20, by = 1),  minsplit = seq(200, 1500, by = 100),  minbucket = seq(20, 500, by = 20),  cp = -1)candidatos$ganancia_pred <- predict(modelo_meta, newdata = candidatos)candidatos <- as.data.table(candidatos)setorder(candidatos, -ganancia_pred)cat("Top 20 combinaciones sugeridas por el metamodelo:\n")print(head(candidatos, 20))

In [ ]:
# =============================================================================# 5. Reporte plano de la region donde el metamodelo concentra el maximo# =============================================================================# top 20 combinaciones observadas (media de semillas)cat("Top 20 combinaciones observadas en HT29003:\n")print(tb_combo[1:20, .(cp, maxdepth, minsplit, minbucket, ganancia_mean, ganancia_sd)])# distribucion de maxdepth dentro del top 20 (donde vive el optimo)cat("\nmaxdepth mas frecuente en el top 20:\n")print(table(tb_combo[1:20, maxdepth]))